# Grasp-Anything++ — kiểm tra schema

Trả lời đúng một câu hỏi: **dữ liệu GA++ thật sự trông như thế nào**, để viết loader cho đúng
thay vì đoán.

GA++ nặng ~150 GB nếu tải đủ (65 GB ảnh ở repo base + ~10 GB label ở repo pp). Notebook này
**không cần tải gì cả** ở chế độ mặc định: zip trên HuggingFace hỗ trợ HTTP range request, nên
ta đọc trực tiếp vài chục file mẫu bên trong archive từ xa (~vài MB) rồi decode.

| phần | tải về | mặc định |
|---|---|---|
| §1–§4 peek schema từ xa | ~5 MB | **bật** |
| §5 dựng mini-subset khớp id | ~2.5 GB (quét central directory) | tắt |
| §6 tải full + verify dưới đĩa | ~150 GB | tắt |

Chỉ dùng thư viện chuẩn — `torch`/`numpy` là tuỳ chọn (có thì dùng, không có thì đọc tay).

In [ ]:
import os
from pathlib import Path

# Notebook có thể chạy từ notebooks/ hoặc từ repo root -> tự tìm root.
ROOT = Path.cwd()
while not (ROOT / "utils" / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("repo root:", ROOT)

REPO_BASE = "airvlab/Grasp-Anything"       # ảnh + scene_description (object-level)
REPO_PP   = "airvlab/Grasp-Anything-pp"    # prompt + part_mask + grasp label (part-level)

DATA_DIR  = Path("data/grasp-anything-pp")        # đích của script/download_grasp_anything_pp.sh
MINI_DIR  = Path("data/grasp-anything-pp-mini")   # subset nhỏ dựng ở §5
CACHE_DIR = Path("data/_zip_index")               # cache central directory

N_PEEK            = 6      # số file mẫu decode ở §3
N_STAT            = 4000   # số file mẫu (chỉ tên) dùng cho thống kê id ở §4
BUILD_MINI        = False  # §5 — quét central directory (~2.5 GB), dựng subset khớp id
MINI_SAMPLES      = 32
RUN_FULL_DOWNLOAD = False  # §6 — gọi script tải ~150 GB

## 0. Helper: đọc zip từ xa qua HTTP range

Ba thứ cần thiết:

- `Archive.stream_members()` — đi tuần tự từ đầu archive theo *local file header*, lấy N file đầu.
  Rẻ nhất, đủ để xem schema (không cần central directory).
- `Archive.central_dir()` — stream toàn bộ central directory để liệt kê **mọi** entry. Đắt
  (670–715 MB mỗi archive GA++) nên chỉ dùng khi §5 bật.
- `Archive.read_member()` — random access một file bất kỳ, cần entry từ central directory.

`image.zip` bị chẻ làm hai file (`image_part_aa` + `image_part_ab`) và offset trong zip tính trên
bản đã nối, nên `Archive` nhận nhiều part và tự map offset logic → (part, offset thật).

In [ ]:
import struct
import urllib.request
import zlib

HF = "https://huggingface.co/datasets/{repo}/resolve/main/{name}"


def _range_get(url, start, end):
    """Đọc byte [start, end] (inclusive). Trả (data, content_range)."""
    req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return r.read(), r.headers.get("Content-Range")


class Archive:
    """Một zip trên HuggingFace, đọc qua HTTP range. Nhiều `names` = các part nối lại."""

    def __init__(self, repo, *names):
        self.repo, self.names = repo, names
        self.urls = [HF.format(repo=repo, name=n) for n in names]
        self.sizes = [int(_range_get(u, 0, 0)[1].split("/")[1]) for u in self.urls]
        self.size = sum(self.sizes)

    def __repr__(self):
        return f"<Archive {self.repo}/{'+'.join(self.names)} {self.size / 1e9:.2f} GB>"

    def read_at(self, start, end):
        """Byte [start, end] trên archive logic (đã nối part)."""
        out, base = b"", 0
        for url, n in zip(self.urls, self.sizes):
            lo, hi = base, base + n - 1
            if start <= hi and end >= lo:
                out += _range_get(url, max(start, lo) - base, min(end, hi) - base)[0]
            base += n
        return out

    # ------------------------------------------------------------- EOCD --
    def eocd(self):
        """(n_entries, cd_size, cd_offset). Mọi archive ở đây > 65535 file nên đều là ZIP64."""
        tail = self.read_at(max(0, self.size - 65536), self.size - 1)
        i = tail.rfind(b"PK\x05\x06")
        if i < 0:
            raise ValueError("không thấy End Of Central Directory")
        n, cd_size, cd_off = struct.unpack("<HII", tail[i + 10 : i + 20])
        j = tail.rfind(b"PK\x06\x07")  # ZIP64 EOCD locator
        if j >= 0:
            z64_off = struct.unpack("<Q", tail[j + 8 : j + 16])[0]
            z = self.read_at(z64_off, z64_off + 55)
            if z[:4] == b"PK\x06\x06":
                n, cd_size, cd_off = struct.unpack("<QQQ", z[32:56])
        return n, cd_size, cd_off

    # ------------------------------------------- đi tuần tự từ đầu file --
    def stream_members(self, max_members=8, max_bytes=8 << 20, decompress=True):
        """N file đầu tiên theo thứ tự lưu trong archive. Trả [(name, usize, data)]."""
        buf = self.read_at(0, max_bytes - 1)
        out, p = [], 0
        while len(out) < max_members and p + 30 <= len(buf):
            if buf[p : p + 4] != b"PK\x03\x04":
                break
            flags, method = struct.unpack("<HH", buf[p + 6 : p + 10])
            csize, usize, nlen, elen = struct.unpack("<IIHH", buf[p + 18 : p + 30])
            name = buf[p + 30 : p + 30 + nlen].decode("utf-8", "replace")
            extra = buf[p + 30 + nlen : p + 30 + nlen + elen]
            csize, usize, _ = _zip64_fix(extra, csize, usize, 0)
            start = p + 30 + nlen + elen
            if flags & 0x08 and csize == 0:
                break  # data descriptor: kích thước chỉ biết sau khi đọc hết, không đi tiếp được
            data = buf[start : start + csize]
            if len(data) < csize:
                break
            if not name.endswith("/"):
                if decompress and method == 8:
                    data = zlib.decompressobj(-15).decompress(data)
                out.append((name, usize, data))
            p = start + csize
        return out

    # ------------------------------------------------ central directory --
    def central_dir(self, limit=None, chunk=8 << 20, progress=None):
        """Yield dict(name, csize, usize, method, off) cho từng entry."""
        n_entries, cd_size, cd_off = self.eocd()
        end, pos, buf, seen = cd_off + cd_size, cd_off, b"", 0
        while pos < end:
            stop = min(pos + chunk, end)
            buf += self.read_at(pos, stop - 1)
            pos = stop
            p = 0
            while len(buf) - p >= 46 and buf[p : p + 4] == b"PK\x01\x02":
                method = struct.unpack("<H", buf[p + 10 : p + 12])[0]
                csize, usize = struct.unpack("<II", buf[p + 20 : p + 28])
                nlen, elen, clen = struct.unpack("<HHH", buf[p + 28 : p + 34])
                if len(buf) - p < 46 + nlen + elen + clen:
                    break  # entry bị cắt giữa chừng -> chờ chunk sau
                lho = struct.unpack("<I", buf[p + 42 : p + 46])[0]
                name = buf[p + 46 : p + 46 + nlen].decode("utf-8", "replace")
                extra = buf[p + 46 + nlen : p + 46 + nlen + elen]
                csize, usize, lho = _zip64_fix(extra, csize, usize, lho)
                p += 46 + nlen + elen + clen
                seen += 1
                if not name.endswith("/"):
                    yield dict(name=name, csize=csize, usize=usize, method=method, off=lho)
                if progress and seen % progress == 0:
                    print(f"  ... {seen:,}/{n_entries:,} entries", end="\r")
                if limit and seen >= limit:
                    return
            buf = buf[p:]

    def read_member(self, entry, decompress=True):
        """Đọc một file, `entry` lấy từ central_dir()."""
        if entry["csize"] == 0:
            return b""
        head = self.read_at(entry["off"], entry["off"] + 29)
        nlen, elen = struct.unpack("<HH", head[26:30])
        start = entry["off"] + 30 + nlen + elen
        raw = self.read_at(start, start + entry["csize"] - 1)
        if decompress and entry["method"] == 8:
            return zlib.decompressobj(-15).decompress(raw)
        return raw


def _zip64_fix(extra, csize, usize, lho):
    """Field 0xFFFFFFFF nghĩa là giá trị thật nằm trong extra field ZIP64 (header id 0x0001)."""
    if 0xFFFFFFFF not in (csize, usize, lho):
        return csize, usize, lho
    q = 0
    while q + 4 <= len(extra):
        hid, hsz = struct.unpack("<HH", extra[q : q + 4])
        if hid == 0x0001:
            f, k = extra[q + 4 : q + 4 + hsz], 0
            if usize == 0xFFFFFFFF:
                usize = struct.unpack("<Q", f[k : k + 8])[0]; k += 8
            if csize == 0xFFFFFFFF:
                csize = struct.unpack("<Q", f[k : k + 8])[0]; k += 8
            if lho == 0xFFFFFFFF:
                lho = struct.unpack("<Q", f[k : k + 8])[0]
            break
        q += 4 + hsz
    return csize, usize, lho

### Decode từng định dạng

`.pkl` là pickle thường. `.npy` có header text đọc tay được. `.pt` là zip do `torch.save()` tạo:
`archive/data.pkl` + storage nhị phân — unpickle với stub cho `torch.*` là lấy được dtype/shape/giá
trị mà **không cần cài torch** (tiện khi chỉ muốn xem schema).

In [ ]:
import array
import ast
import io
import pickle
import zipfile

_TORCH_STORAGE = {  # tên storage class -> (typecode của array, itemsize)
    "FloatStorage": ("f", 4), "DoubleStorage": ("d", 8), "LongStorage": ("q", 8),
    "IntStorage": ("i", 4), "ShortStorage": ("h", 2), "CharStorage": ("b", 1),
    "ByteStorage": ("B", 1), "BoolStorage": ("B", 1), "HalfStorage": (None, 2),
}


class _Stub:
    """Thay cho mọi class torch.* khi unpickle: chỉ ghi lại tên + tham số."""

    def __init__(self, name):
        self.name = name

    def __call__(self, *a, **k):
        return (self.name, a, k)

    def __reduce__(self):
        return (_Stub, (self.name,))


class _StubUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        return _Stub(f"{module}.{name}")

    def persistent_load(self, pid):
        return pid  # ('storage', <dtype class>, key, location, numel)


def read_pt(raw):
    """Đọc tensor từ bytes của một file .pt. -> dict(dtype, shape, rows)."""
    try:
        import torch  # noqa: F401 — có torch thì dùng cho chắc
        t = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)
        return dict(dtype=str(t.dtype), shape=tuple(t.shape), rows=t.tolist())
    except Exception:
        pass
    zf = zipfile.ZipFile(io.BytesIO(raw))
    pkl_name = next(n for n in zf.namelist() if n.endswith("data.pkl"))
    prefix = pkl_name[: -len("data.pkl")]
    _fn, args, _kw = _StubUnpickler(io.BytesIO(zf.read(pkl_name))).load()
    (_tag, dtype_stub, key, _loc, numel), _offset, size, _stride = args[:4]
    dtype = dtype_stub.name.split(".")[-1]
    typecode, itemsize = _TORCH_STORAGE.get(dtype, (None, None))
    rows = None
    if typecode:
        a = array.array(typecode)
        a.frombytes(zf.read(f"{prefix}data/{key}")[: numel * itemsize])
        flat, ncol = list(a), (size[1] if len(size) > 1 else 1)
        rows = [flat[i : i + ncol] for i in range(0, len(flat), ncol)]
    return dict(dtype=dtype.replace("Storage", "").lower(), shape=tuple(size), rows=rows)


def read_npy(raw):
    """Đọc .npy uint8/nhỏ mà không cần numpy. -> dict(dtype, shape, counts)."""
    assert raw[:6] == b"\x93NUMPY", "không phải .npy"
    major = raw[6]
    hlen_size = 2 if major == 1 else 4
    hlen = int.from_bytes(raw[8 : 8 + hlen_size], "little")
    header = ast.literal_eval(raw[8 + hlen_size : 8 + hlen_size + hlen].decode().strip())
    body = raw[8 + hlen_size + hlen :]
    counts = None
    if header["descr"] in ("|u1", "|b1", "|i1"):
        from collections import Counter
        counts = dict(sorted(Counter(body).items()))
    return dict(dtype=header["descr"], shape=header["shape"], counts=counts, nbytes=len(body))


def read_jpeg_size(raw):
    """(width, height) từ SOF marker — không cần Pillow."""
    i = 2
    while i < len(raw) - 9:
        if raw[i] != 0xFF:
            i += 1
            continue
        marker, seglen = raw[i + 1], int.from_bytes(raw[i + 2 : i + 4], "big")
        if 0xC0 <= marker <= 0xCF and marker not in (0xC4, 0xC8, 0xCC):
            h = int.from_bytes(raw[i + 5 : i + 7], "big")
            w = int.from_bytes(raw[i + 7 : i + 9], "big")
            return w, h
        i += 2 + seglen
    return None

## 1. Có những file gì trên HuggingFace

Cả hai repo đều `gated: false` — tải được không cần token. (Tác giả vẫn đề nghị điền form ở
<https://airvlab.github.io/grasp-anything/docs/download/> trước khi dùng.)

In [ ]:
import json
import urllib.request

def list_repo(repo):
    url = f"https://huggingface.co/api/datasets/{repo}?blobs=true"
    with urllib.request.urlopen(url, timeout=60) as r:
        return json.load(r)

for repo in (REPO_BASE, REPO_PP):
    info = list_repo(repo)
    print(f"== {repo}   gated={info.get('gated')}  private={info.get('private')}")
    for s in sorted(info["siblings"], key=lambda s: -s.get("size", 0)):
        if s.get("size", 0) > 1e6:
            print(f"   {s['size'] / 1e9:7.2f} GB  {s['rfilename']}")

## 2. Kích thước / số entry của từng archive

Đọc EOCD ở cuối mỗi zip (~vài chục KB) là biết số file bên trong, không cần tải.

In [ ]:
ARCHIVES = {
    "image":                 Archive(REPO_BASE, "image_part_aa", "image_part_ab"),
    "scene_description":     Archive(REPO_BASE, "scene_description.zip"),
    "base grasp_label_pos":  Archive(REPO_BASE, "grasp_label_positive.zip"),
    "pp grasp_instructions": Archive(REPO_PP, "grasp_instructions.zip"),
    "pp grasp_label_pos":    Archive(REPO_PP, "grasp_label_positive.zip"),
    "pp part_mask":          Archive(REPO_PP, "part_mask.zip"),
    "pp grasp_label_neg":    Archive(REPO_PP, "grasp_label_negative.zip"),
}

print(f"{'archive':24} {'size':>9} {'entries':>12} {'central dir':>12}")
stats = {}
for label, arc in ARCHIVES.items():
    n, cd_size, _ = arc.eocd()
    stats[label] = n
    print(f"{label:24} {arc.size / 1e9:6.2f} GB {n:12,} {cd_size / 1e6:9.0f} MB")

Bốn archive GA++ có **cùng số entry** → nhiều khả năng id khớp 1:1 giữa `grasp_instructions/`,
`grasp_label_positive/`, `grasp_label_negative/` và `part_mask/`. (§5/§6 kiểm chứng thật.)

`image` và `scene_description` thì ở mức *scene*, ít hơn nhiều — một ảnh dùng chung cho mọi
sample part-level của nó.

## 3. Schema thật của từng thư mục

Lấy vài file đầu của mỗi archive rồi decode.

In [ ]:
peek = {label: arc.stream_members(max_members=N_PEEK, max_bytes=6 << 20)
        for label, arc in ARCHIVES.items()}
for label, members in peek.items():
    print(f"{label:24} {len(members)} mẫu, vd: {members[0][0]}")

### 3.1 `image/` — ảnh RGB, tên là SHA-256 của scene

In [ ]:
for name, usize, data in peek["image"][:3]:
    print(f"{name:>80}  {usize / 1024:6.1f} KB  jpeg {read_jpeg_size(data)}")

### 3.2 `scene_description/` (base GA) — caption **cả scene**, không phải prompt gắp

In [ ]:
for name, usize, data in peek["scene_description"][:3]:
    obj = pickle.loads(data)
    print(name.split("/")[-1])
    print(f"   type={type(obj).__name__} len={len(obj)}  ->  {obj}")

### 3.3 `grasp_instructions/` (GA++) — prompt gắp, **một string mỗi part**

Đây là thứ thay thế `scene_description/` cho task language-driven: mỗi file là một câu lệnh gắp
nhắm vào **một part cụ thể** của một object. Không phải list, không phải tuple — pickle của một
`str` thuần.

In [ ]:
from collections import Counter

for name, usize, data in peek["pp grasp_instructions"]:
    obj = pickle.loads(data)
    print(f"{name.split('/')[-1]:>75}  {type(obj).__name__:>4}  {obj!r}")

types = Counter(type(pickle.loads(d)).__name__ for _, _, d in peek["pp grasp_instructions"])
print("\nkiểu dữ liệu:", dict(types))

### 3.4 `grasp_label_positive/` — grasp part-level, `(N, 6)` float32

Layout **giống hệt base GA**: mỗi hàng là `[q, x, y, w, h, theta_deg]`, góc tính bằng độ. Cột `q`
là điểm antipodal `T̃ = (cos α₁ + cos α₂) / R` mà paper LGD (§3.2 Grasp Annotation) dùng để phân
dương/âm: `T̃ > 0` vào `grasp_label_positive/`, còn lại vào `grasp_label_negative/` — khớp với số
đo ở §3.6 dưới đây (cột 0 xấp xỉ 0 hoặc âm). `_grasp_anything_format` bỏ qua cột này. Nghĩa là
[`load_from_grasp_anything_file`](utils/dataset_processing/grasp.py) và
[`_grasp_anything_format`](utils/dataset_processing/grasp.py) dùng lại được nguyên vẹn cho GA++.

In [ ]:
print("-- GA++ (part-level) --")
for name, _, data in peek["pp grasp_label_pos"][:3]:
    t = read_pt(data)
    print(f"{name.split('/')[-1]:>75}  {t['dtype']} {t['shape']}")
    for row in (t["rows"] or [])[:2]:
        print("      ", "  ".join(f"{v:8.3f}" for v in row))

print("\n-- base GA (object-level), để so sánh --")
for name, _, data in peek["base grasp_label_pos"][:2]:
    t = read_pt(data)
    print(f"{name.split('/')[-1]:>75}  {t['dtype']} {t['shape']}")
    for row in (t["rows"] or [])[:2]:
        print("      ", "  ".join(f"{v:8.3f}" for v in row))

### 3.5 `part_mask/` — mask nhị phân 416×416 uint8

Giá trị `{0, 1}` (không phải 0/255), cùng kích thước ảnh gốc nên chịu chung phép rot/zoom với ảnh
là được.

In [ ]:
for name, usize, data in peek["pp part_mask"][:3]:
    a = read_npy(data)
    fg = a["counts"].get(1, 0) / a["nbytes"] * 100 if a["counts"] else float("nan")
    print(f"{name.split('/')[-1]:>75}  {a['dtype']} {a['shape']}  values={a['counts']}  fg={fg:.2f}%")

### 3.6 `grasp_label_negative/` — cùng layout, cột `q` không dương

Đây là mặt kia của ngưỡng `T̃ > 0`: cùng scene/object/part, nhưng là các grasp bị loại.

In [ ]:
for name, _, data in peek["pp grasp_label_neg"][:2]:
    t = read_pt(data)
    print(f"{name.split('/')[-1]:>75}  {t['dtype']} {t['shape']}")
    for row in (t["rows"] or [])[:2]:
        print("      ", "  ".join(f"{v:8.3f}" for v in row))

## 4. Quy ước id — chỗ loader hiện tại sẽ gãy

- base GA: `<scene_sha256>_<object_idx>` — **2 phần**
- GA++:    `<scene_sha256>_<object_idx>_<part_idx>` — **3 phần**
- `split/grasp-anything/{seen,unseen}.obj`: list id **2 phần** (object-level)
- `image/`, `scene_description/`: khoá chỉ là `<scene_sha256>`

Lấy một mẫu tên file lớn hơn (chỉ tên, không decode) để thống kê.

In [ ]:
arc_instr = ARCHIVES["pp grasp_instructions"]
sample = arc_instr.stream_members(max_members=N_STAT, max_bytes=8 << 20, decompress=False)
ids = [Path(n).stem for n, _, _ in sample]

scenes = {i.rsplit("_", 2)[0] for i in ids}
objects = {i.rsplit("_", 1)[0] for i in ids}
print(f"{len(ids):,} sample  ->  {len(objects):,} object id  ->  {len(scenes):,} scene id")
print("số phần trong id:", Counter(len(i.split('_')) for i in ids))
print("ví dụ:", ids[:3])

In [ ]:
seen = set(pickle.load(open("split/grasp-anything/seen.obj", "rb")))
unseen = set(pickle.load(open("split/grasp-anything/unseen.obj", "rb")))
print(f"split seen={len(seen):,}  unseen={len(unseen):,}  (id 2 phần, vd {next(iter(seen))})")

hit_raw = sum(i in seen or i in unseen for i in ids)
hit_obj = sum(i.rsplit("_", 1)[0] in seen or i.rsplit("_", 1)[0] in unseen for i in ids)
print(f"\nid GA++ nguyên vẹn khớp split : {hit_raw:5d} / {len(ids)}   <- loader hiện tại lọc kiểu này")
print(f"id GA++ bỏ part_idx khớp split : {hit_obj:5d} / {len(ids)}")

`hit_raw == 0` là kết luận quan trọng nhất: [`GraspAnythingDataset`](utils/data/grasp_anything_data.py)
lọc bằng `self._sample_id(x) in idxs` với `_sample_id` = tên file đầy đủ, nên chạy trên thư mục GA++
sẽ ra **0 file** rồi `FileNotFoundError('No dataset files found')` — trông như sai đường dẫn, thực ra
là sai quy ước id.

Lỗi thứ hai, âm thầm hơn:

In [ ]:
import re

src = Path("utils/data/grasp_anything_data.py").read_text()
pattern = re.search(r'GRASP_SUFFIX_RE = re\.compile\(r"([^"]+)"\)', src).group(1)
sample_id = ids[0]
print("regex hiện tại :", pattern)
print("id GA++        :", sample_id)
print("scene_id suy ra:", re.sub(pattern, "", sample_id + ".pt"))
print("scene_id đúng  :", sample_id.rsplit("_", 2)[0])
print("\n-> get_rgb_file() sẽ trỏ tới image/<scene>_<obj>.jpg, file không tồn tại.")

**Tóm tắt cho loader GA++** (`utils/data/grasp_anything_pp_data.py`):

| việc | dùng lại được? |
|---|---|
| `GraspRectangles.load_from_grasp_anything_file()` | ✅ layout `(N,6)` y hệt |
| `_sample_id()` / `GRASP_SUFFIX_RE` | ❌ phải bỏ **hai** hậu tố để ra `<scene>` |
| lọc theo `split/grasp-anything/*.obj` | ❌ phải map `<scene>_<obj>_<part>` → `<scene>_<obj>` trước khi so |
| `scene_description/` | ❌ thay bằng `grasp_instructions/` (một `str`, không phải tuple) |
| `part_mask/` | mới — 416×416 uint8 `{0,1}`, phải chịu cùng rot/zoom với ảnh |

## 5. (tuỳ chọn) Dựng mini-subset khớp id để code loader

Muốn chạy thử loader mà không tải 150 GB thì cần một thư mục nhỏ có **đủ 4 thành phần cho cùng
một id**. Thứ tự lưu trong các archive khác nhau nên không thể lấy N file đầu của mỗi cái rồi ghép
— phải tra central directory.

Chi phí: ~2.5 GB range read (CD của 3 archive GA++ + CD ảnh), một lần, có cache dưới `data/_zip_index/`.
Bật bằng `BUILD_MINI = True` ở cell config.

In [ ]:
import gzip

def cached_index(label, arc, prefix):
    """{id -> entry} cho mọi file trong archive. Cache tên+offset xuống đĩa (một lần)."""
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / f"{label.replace(' ', '_')}.tsv.gz"
    if path.exists():
        index = {}
        with gzip.open(path, "rt") as f:
            for line in f:
                base, off, csize, usize, method = line.rstrip("\n").split("\t")
                index[Path(base).stem] = dict(name=prefix + base, off=int(off), csize=int(csize),
                                              usize=int(usize), method=int(method))
        print(f"[cache] {label}: {len(index):,} entry")
        return index

    print(f"[scan ] {label}: đọc central directory (vài phút)...")
    index = {}
    with gzip.open(path.with_suffix(".part.gz"), "wt") as f:
        for e in arc.central_dir(progress=250_000):
            base = Path(e["name"]).name
            index[Path(base).stem] = e
            f.write(f"{base}\t{e['off']}\t{e['csize']}\t{e['usize']}\t{e['method']}\n")
    path.with_suffix(".part.gz").rename(path)
    print(f"\n[scan ] {label}: {len(index):,} entry")
    return index


if BUILD_MINI:
    idx_instr = cached_index("pp grasp_instructions", ARCHIVES["pp grasp_instructions"], "grasp_instructions/")
    idx_label = cached_index("pp grasp_label_pos", ARCHIVES["pp grasp_label_pos"], "grasp_label_positive/")
    idx_mask  = cached_index("pp part_mask", ARCHIVES["pp part_mask"], "part_mask/")
    idx_image = cached_index("image", ARCHIVES["image"], "image/")

    common = set(idx_instr) & set(idx_label) & set(idx_mask)
    print(f"\nid có đủ prompt + label + mask: {len(common):,} / {len(idx_instr):,}")
    orphan_scene = {i.rsplit("_", 2)[0] for i in list(common)[:100_000]} - set(idx_image)
    print(f"scene thiếu ảnh (trong 100k mẫu): {len(orphan_scene)}")
else:
    print("BUILD_MINI = False -> bỏ qua (đặt True nếu muốn dựng subset).")

In [ ]:
if BUILD_MINI:
    import random

    random.seed(0)
    picked = sorted(random.sample(sorted(common), MINI_SAMPLES))
    for sub in ("image", "grasp_instructions", "grasp_label_positive", "part_mask"):
        (MINI_DIR / sub).mkdir(parents=True, exist_ok=True)

    for k, sid in enumerate(picked, 1):
        scene = sid.rsplit("_", 2)[0]
        (MINI_DIR / "grasp_instructions" / f"{sid}.pkl").write_bytes(
            ARCHIVES["pp grasp_instructions"].read_member(idx_instr[sid]))
        (MINI_DIR / "grasp_label_positive" / f"{sid}.pt").write_bytes(
            ARCHIVES["pp grasp_label_pos"].read_member(idx_label[sid]))
        (MINI_DIR / "part_mask" / f"{sid}.npy").write_bytes(
            ARCHIVES["pp part_mask"].read_member(idx_mask[sid]))
        img = MINI_DIR / "image" / f"{scene}.jpg"
        if not img.exists():
            img.write_bytes(ARCHIVES["image"].read_member(idx_image[scene]))
        print(f"  [{k}/{len(picked)}] {sid}", end="\r")

    print(f"\nxong -> {MINI_DIR}")
    for sub in sorted(p.name for p in MINI_DIR.iterdir()):
        print(f"   {sub:22} {len(list((MINI_DIR / sub).iterdir())):3d} file")

## 6. (tuỳ chọn) Tải full rồi verify dưới đĩa

`script/download_grasp_anything_pp.sh` tải ảnh (base repo) + 3 thư mục label (GA++) vào một chỗ.
~150 GB, chạy nhiều giờ — nên chạy trong terminal chứ không phải trong notebook. Cell dưới chỉ để
tiện, và bị khoá sau `RUN_FULL_DOWNLOAD`.

In [ ]:
import subprocess

if RUN_FULL_DOWNLOAD:
    subprocess.run(["bash", "script/download_grasp_anything_pp.sh", "--dest", str(DATA_DIR)], check=True)
else:
    print("RUN_FULL_DOWNLOAD = False")
    print("chạy tay:  script/download_grasp_anything_pp.sh --dest", DATA_DIR)
    print("kiểm tra:  script/download_grasp_anything_pp.sh --dest", DATA_DIR, "--check")

### 6.1 Verify dữ liệu dưới đĩa (chạy được cả trên `MINI_DIR`)

In [ ]:
VERIFY_DIR = MINI_DIR if MINI_DIR.exists() else DATA_DIR

def verify(root):
    root = Path(root)
    if not root.exists():
        print(f"{root} chưa tồn tại — chạy §5 hoặc §6 trước.")
        return
    print(f"== {root}")
    folders = {p.name: p for p in root.iterdir() if p.is_dir() and not p.name.startswith("_")}
    counts = {}
    for name, path in sorted(folders.items()):
        files = list(path.iterdir())
        counts[name] = {f.stem for f in files}
        print(f"   {name:22} {len(files):9,} file   vd {files[0].name}")

    if "grasp_instructions" in counts and "grasp_label_positive" in counts:
        a, b = counts["grasp_instructions"], counts["grasp_label_positive"]
        print(f"\n   prompt \\ label = {len(a - b):,}   label \\ prompt = {len(b - a):,}")
    if "part_mask" in counts and "grasp_label_positive" in counts:
        a, b = counts["part_mask"], counts["grasp_label_positive"]
        print(f"   mask   \\ label = {len(a - b):,}   label \\ mask   = {len(b - a):,}")
    if "image" in counts and "grasp_label_positive" in counts:
        need = {i.rsplit("_", 2)[0] for i in counts["grasp_label_positive"]}
        print(f"   scene cần ảnh = {len(need):,}, thiếu = {len(need - counts['image']):,}")

    sids = sorted(counts.get("grasp_label_positive", []))
    if sids:
        per_scene = Counter(i.rsplit("_", 2)[0] for i in sids)
        per_obj = Counter(i.rsplit("_", 1)[0] for i in sids)
        print(f"\n   sample/scene: trung bình {len(sids) / len(per_scene):.2f}, max {max(per_scene.values())}")
        print(f"   part/object : trung bình {len(sids) / len(per_obj):.2f}, max {max(per_obj.values())}")

verify(VERIFY_DIR)

### 6.2 Đọc thử một sample hoàn chỉnh

In [ ]:
root = Path(VERIFY_DIR)
if (root / "grasp_label_positive").exists():
    sid = sorted(p.stem for p in (root / "grasp_label_positive").iterdir())[0]
    scene = sid.rsplit("_", 2)[0]

    prompt = pickle.loads((root / "grasp_instructions" / f"{sid}.pkl").read_bytes())
    grasps = read_pt((root / "grasp_label_positive" / f"{sid}.pt").read_bytes())
    mask = read_npy((root / "part_mask" / f"{sid}.npy").read_bytes())
    img = (root / "image" / f"{scene}.jpg").read_bytes()

    print("sample id :", sid)
    print("prompt    :", repr(prompt))
    print("image     :", read_jpeg_size(img), f"{len(img) / 1024:.0f} KB")
    print("part_mask :", mask["dtype"], mask["shape"], mask["counts"])
    print("grasps    :", grasps["dtype"], grasps["shape"], "[_, x, y, w, h, theta_deg]")
    for row in (grasps["rows"] or [])[:5]:
        print("     ", "  ".join(f"{v:8.3f}" for v in row))
else:
    print("chưa có dữ liệu dưới đĩa.")

### 6.3 Vẽ thử: ảnh + part_mask + grasp rectangle (cần numpy/Pillow/matplotlib)

`grasp_rectangle()` chép đúng công thức của
[`Grasp.as_gr`](utils/dataset_processing/grasp.py) + [`_grasp_anything_format`](utils/dataset_processing/grasp.py)
— toạ độ `[y, x]`, góc lật dấu, `w` là chiều dài giữa hai ngón, `h` là bề rộng kẹp. Vẽ ra khớp với
object trong ảnh nghĩa là cách diễn giải 6 cột là đúng cho GA++.

In [ ]:
import math

def grasp_rectangle(row):
    """Một hàng (N,6) của GA/GA++ -> 4 đỉnh (x, y) để vẽ."""
    _, x, y, w, h, theta_deg = row
    a = -math.radians(theta_deg)           # GA lật góc quanh trục ngang
    xo, yo = math.cos(a), math.sin(a)
    y1, x1 = y + w / 2 * yo, x - w / 2 * xo
    y2, x2 = y - w / 2 * yo, x + w / 2 * xo
    pts = [(x1 - h / 2 * yo, y1 - h / 2 * xo),
           (x2 - h / 2 * yo, y2 - h / 2 * xo),
           (x2 + h / 2 * yo, y2 + h / 2 * xo),
           (x1 + h / 2 * yo, y1 + h / 2 * xo)]
    return pts + pts[:1]


try:
    import io as _io

    import matplotlib.pyplot as plt
    import numpy as np
    from PIL import Image

    rgb = np.array(Image.open(_io.BytesIO(img)))
    m = np.load(root / "part_mask" / f"{sid}.npy")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    axes[0].imshow(rgb)
    axes[0].set_title(f"image {scene[:12]}…")
    axes[1].imshow(m, cmap="gray")
    axes[1].set_title(f"part_mask  fg={m.mean() * 100:.1f}%")
    axes[2].imshow(rgb)
    axes[2].imshow(np.ma.masked_where(m == 0, m), alpha=0.4, cmap="autumn")
    for row in (grasps["rows"] or [])[:8]:
        xs, ys = zip(*grasp_rectangle(row))
        axes[2].plot(xs, ys, lw=1.2)
    axes[2].set_title(repr(prompt)[:48], fontsize=9)
    for a in axes:
        a.axis("off")
    plt.tight_layout()
    plt.show()
except ImportError as e:
    print("thiếu thư viện vẽ (numpy/Pillow/matplotlib):", e)
except NameError:
    print("chạy cell 6.2 trước.")

## 7. Kết luận

Schema GA++ (đã verify trực tiếp trên archive, không phải đọc README):

```
<root>/image/<scene>.jpg                        416×416 RGB, ~995k scene, dùng chung
      /grasp_instructions/<scene>_<obj>_<part>.pkl   pickle của MỘT str
      /grasp_label_positive/<scene>_<obj>_<part>.pt  float32 (N, 6) = [q, x, y, w, h, theta_deg]
      /grasp_label_negative/<scene>_<obj>_<part>.pt  cùng layout, q <= 0
      /part_mask/<scene>_<obj>_<part>.npy            uint8 (416, 416), giá trị {0, 1}
```

4.412.384 sample part-level trên ~994.860 scene (≈4.4 sample/scene).

Việc còn phải làm cho `utils/data/grasp_anything_pp_data.py`:

1. `scene_id = sid.rsplit("_", 2)[0]` (bỏ **hai** hậu tố), `object_id = sid.rsplit("_", 1)[0]`.
2. Lọc split bằng `object_id`, không phải `sid` — nếu không sẽ ra 0 file. Cân nhắc dựng split riêng
   `split/grasp-anything-pp/` vì split hiện tại chỉ phủ 23.098 object id.
3. Prompt đọc từ `grasp_instructions/` là `str` thuần — khác `scene_description/` (tuple
   `(caption, [objects])`).
4. `part_mask` phải đi qua **cùng** rot/zoom/resize với ảnh và grasp label.
5. `GraspDatasetBase.__getitem__` hiện trả `x, (pos, cos, sin, width), idx, rot, zoom` — chưa có chỗ
   cho prompt và mask; mở rộng chữ ký này kéo theo `inference/models/grasp_model.py`.

Ngoài lề, gặp khi viết notebook này: `Grasp.as_gr` dùng `np.float`, đã bị gỡ khỏi numpy ≥ 1.24 —
`utils/dataset_processing/grasp.py` sẽ `AttributeError` trên môi trường numpy mới.